In [ ]:
import subprocess, sys, os


os.chdir(os.path.expanduser("~/cdg2"))
print("cwd:", os.getcwd())

for pkg in ["matplotlib", "scikit-learn", "scipy"]:
    try:
        __import__(pkg.replace("-","_").split(".")[0])
    except ImportError:
        print(f"installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("deps ok")
print("python:", sys.executable)

: 

# CDG2 — Phase 2/3/4 Interactive Analysis

run Phase 1（`run_record.py`），`outputs/manifest.jsonl` 

```
python run_record.py --backend llada_attack \
    --prompt-root prompts/cdg_injection --sae-root ./saes \
    --judge --judge-groups B --out outputs
```

In [ ]:
# ── 0. configs ─────────────────────────────────────────────────────────────────
RECORDS_DIR  = "./outputs"       
SAE_ROOT     = "./saes"         
MODEL_ID     = "GSAI-ML/LLaDA-8B-Instruct"
BACKEND      = "llada_attack"    


FOCUS_LAYER  = 16
FOCUS_FRAC   = 0.10
FOCUS_SCOPE  = "tpl_mask"       
TOP_K        = 20               

In [ ]:
# ── 1. load records ─────────────────────────────────────────────────────────────
from cdg.probe import load_records
from cdg.config import get_backend_config

cfg = get_backend_config(BACKEND)
records = load_records(RECORDS_DIR, cfg.name)
print(f"loaded {len(records)} records")

from cdg.probe import group_letter
from collections import Counter
cnt = Counter(group_letter(r) for r in records)
print("groups:", dict(sorted(cnt.items())))

# see judge results of B group
b_records = [r for r in records if group_letter(r) == "B"]
for r in b_records:
    j = r.get("judge") or {}
    print(f"  {r['case_id']}  success={j.get('success')}  label={j.get('label')}")

## Phase 2-A：linear prove find best layer

higher auc means more worth to do it

In [ ]:
from cdg.probe import probe_sweep

sweep = probe_sweep(
    records,
    scopes=["tpl_mask", "tpl_ctx", "out_unmask"],
    fracs=[0.05, 0.10, 0.20, 0.35, 0.50, 1.00],
    layers=list(cfg.record_layers),   # [11, 16, 26]
    space="hidden",
    pos_groups=("B",), neg_groups=("C",),
)

sweep.sort(key=lambda r: r.get("auc") or 0, reverse=True)
print(f"{'scope':12} {'frac':6} {'layer':6} {'AUC':8} {'F1':8} {'n':4}")
print("-" * 50)
for r in sweep[:10]:
    auc = r.get('auc', float('nan'))
    f1  = r.get('macro_f1', float('nan'))
    print(f"{r['scope']:12} {r['frac']:6.2f} {r['layer']:6d} {auc:8.3f} {f1:8.3f} {r.get('n',0):4d}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

scopes_plot = ["tpl_mask", "tpl_ctx", "out_unmask"]
layers_plot  = sorted(cfg.record_layers)
fracs_plot   = sorted({r["frac"] for r in sweep})

fig, axes = plt.subplots(1, len(scopes_plot), figsize=(5*len(scopes_plot), 4), sharey=True)
for ax, sc in zip(axes, scopes_plot):
    mat = np.full((len(fracs_plot), len(layers_plot)), float('nan'))
    for r in sweep:
        if r["scope"] != sc: continue
        fi = fracs_plot.index(r["frac"])
        li = layers_plot.index(r["layer"])
        mat[fi, li] = r.get("auc") or float('nan')
    im = ax.imshow(mat, vmin=0.5, vmax=1.0, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(len(layers_plot))); ax.set_xticklabels([f"L{l}" for l in layers_plot])
    ax.set_yticks(range(len(fracs_plot))); ax.set_yticklabels([f"{f:.2f}" for f in fracs_plot])
    ax.set_title(sc); ax.set_xlabel("layer"); ax.set_ylabel("frac")
    plt.colorbar(im, ax=ax)
plt.suptitle("Linear Probe AUC: B (harmful) vs C (neutral)", y=1.02)
plt.tight_layout()
plt.savefig("probe_auc_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved probe_auc_heatmap.png")

## Phase 2-B：find the most different sae feature

In [ ]:
from cdg.probe import top_diff_features

diff_rows = top_diff_features(
    records,
    scope=FOCUS_SCOPE,
    pairs=[("B","C"), ("A","B"), ("C","D")],
    k=TOP_K,
    stat="mean_gap",    # or "t_stat" when sample is small
)

# find the row of focus config BvsC
bc_row = next(
    (r for r in diff_rows
     if r["layer"]==FOCUS_LAYER
     and abs(r["frac"]-FOCUS_FRAC)<1e-4
     and r["pair"]=="BvsC"),
    None
)

if bc_row is None:
    print("No data for this (layer, frac, pair). Check FOCUS_LAYER / FOCUS_FRAC.")
else:
    print(f"Layer={bc_row['layer']} frac={bc_row['frac']:.2f} pair={bc_row['pair']}  "
          f"n_B={bc_row['n_pos']} n_C={bc_row['n_neg']}")
    print(f"{'feature':>10}  {'gap':>10}  {'B_mean':>10}  {'C_mean':>10}")
    print("-" * 46)
    for f in bc_row["features"]:
        print(f"{f['feature']:10d}  {f['score']:+10.4f}  {f['pos_mean']:10.4f}  {f['neg_mean']:10.4f}")

In [ ]:
# compare the top features of three pairs at the same (layer, frac)
print("Top features per pair at layer={} frac={}\n".format(FOCUS_LAYER, FOCUS_FRAC))
for pair_name in ["BvsC", "AvsB", "CvsD"]:
    row = next(
        (r for r in diff_rows
         if r["layer"]==FOCUS_LAYER
         and abs(r["frac"]-FOCUS_FRAC)<1e-4
         and r["pair"]==pair_name),
        None
    )
    if row is None:
        print(f"  {pair_name}: no data")
        continue
    feat_ids = [f["feature"] for f in row["features"][:8]]
    print(f"  {pair_name}: {feat_ids}")

## Phase 3：sae feature to word losts


In [ ]:
# 加载 SAE（layer=FOCUS_LAYER，mask bundle）
from cdg.sae import load_sae, sae_ckpt_path
import os

sae_kind = "llada_mask"  # tpl_mask scope 用 mask bundle
ckpt, cfgp = sae_ckpt_path(os.path.join(SAE_ROOT, sae_kind), layer=FOCUS_LAYER, trainer=1)
sae = load_sae(ckpt, config_path=cfgp)
print(f"SAE loaded: d={sae.d_model}  n_features={sae.n_features}  k={sae.k}")

In [ ]:

import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)


from transformers import AutoModelForCausalLM, AutoModel
print("loading model for unembedding (may take a minute)...")
# LLaDA 用 AutoModel，lm_head 通常在 model.lm_head
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True,
                                   torch_dtype=torch.float16)
W_unembed = model.lm_head.weight.detach().float().cpu()
print(f"unembedding: {W_unembed.shape}  (vocab_size x d_model)")
del mode

In [ ]:
from cdg.interpret import feature_vocab_labels, print_feature_report

if bc_row is None:
    raise RuntimeError("Run Phase 2-B first to get bc_row")

top_feature_ids = [f["feature"] for f in bc_row["features"][:15]]
print("Analyzing features:", top_feature_ids)

labels = feature_vocab_labels(
    sae, top_feature_ids, tokenizer, W_unembed,
    topk=12, bottom=True
)
print(print_feature_report(labels, n_tokens=10,
                           header=f"Layer {FOCUS_LAYER} frac={FOCUS_FRAC} BvsC top features"))

In [ ]:
from collections import defaultdict

pair_names = ["BvsC", "AvsB", "CvsD"]
feat_pairs = defaultdict(list)

for pair_name in pair_names:
    row = next(
        (r for r in diff_rows
         if r["layer"]==FOCUS_LAYER
         and abs(r["frac"]-FOCUS_FRAC)<1e-4
         and r["pair"]==pair_name),
        None
    )
    if row:
        for f in row["features"]:
            feat_pairs[f["feature"]].append(pair_name)

shared = [(fid, pairs) for fid, pairs in feat_pairs.items() if len(pairs) > 1]
shared.sort(key=lambda x: -len(x[1]))

print("Features appearing in multiple pair top-k:")
for fid, pairs in shared[:10]:
    print(f"  feature {fid:6d}  in: {pairs}")

## Phase 4-A：Feature Atlas 
if non diagnos close zero means they are orthanganle, otherwise means redundancy

In [ ]:
from cdg.interpret import feature_atlas_data
import numpy as np
import matplotlib.pyplot as plt

# 用三个 pair 的 union feature set
union_feats = sorted(feat_pairs.keys())
print(f"Union feature set: {len(union_feats)} features")

atlas = feature_atlas_data(sae, union_feats)
sim = atlas["cosine_sim"]
order = atlas["order"] or list(range(len(union_feats)))
clusters = atlas["clusters"]

sim_ord = sim[np.ix_(order, order)]
feat_ord = [union_feats[i] for i in order]

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(sim_ord, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(feat_ord))); ax.set_xticklabels(feat_ord, rotation=90, fontsize=7)
ax.set_yticks(range(len(feat_ord))); ax.set_yticklabels(feat_ord, fontsize=7)
ax.set_title(f"Feature Atlas — Decoder Direction Cosine Sim\n(Layer {FOCUS_LAYER}, {FOCUS_SCOPE})")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig("feature_atlas.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved feature_atlas.png")

if clusters:
    print("\nCluster assignments:")
    from collections import defaultdict as DD
    cl = DD(list)
    for fid, cid in zip(union_feats, clusters):
        cl[cid].append(fid)
    for cid in sorted(cl):
        print(f"  cluster {cid}: {cl[cid]}")

## Phase 4-B：Co-activation Matrix — 功能相关性

decoder 余弦相似度是**几何**属性（权重空间）。
共激活矩阵是**功能**属性（实际数据上同时激活）。

二者对比：
- 几何高 + 功能高 → 冗余 feature，可以合并
- 几何低 + 功能高 → 方向不同但总是一起出现 → 可能是「攻击流程」里的协同机制
- 几何高 + 功能低 → 权重重叠但实际不一起火 → 多义 feature

In [ ]:
from cdg.interpret import coactivation_matrix

C_mat = coactivation_matrix(
    records,
    feature_ids=union_feats,
    scope=FOCUS_SCOPE,
    frac=FOCUS_FRAC,
    layer=FOCUS_LAYER,
    groups=("B", "C"),   # 在 B+C 所有数据上算相关
)

C_ord = C_mat[np.ix_(order, order)]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, mat, title in [
    (axes[0], sim_ord,  "Decoder Cosine Sim (geometry)"),
    (axes[1], C_ord,    "Co-activation Pearson (function)"),
]:
    im = ax.imshow(mat, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(len(feat_ord))); ax.set_xticklabels(feat_ord, rotation=90, fontsize=7)
    ax.set_yticks(range(len(feat_ord))); ax.set_yticklabels(feat_ord, fontsize=7)
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
plt.suptitle(f"Feature Atlas vs Co-activation — Layer {FOCUS_LAYER}", y=1.01)
plt.tight_layout()
plt.savefig("feature_atlas_vs_coactivation.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved feature_atlas_vs_coactivation.png")

## Phase 4-C：逐步追踪 — feature 激活随去噪进度的变化

看注入攻击在去噪的「哪个时刻」开始激活关键 feature。
早期（frac=0.05）就激活 → 攻击在扩散早期就「锁定」了有害路径。

In [ ]:
from cdg.probe import stack_group
import numpy as np
import matplotlib.pyplot as plt

# 前5个 BvsC top features 随 frac 的均值激活
focus_feats = [f["feature"] for f in bc_row["features"][:5]] if bc_row else []

all_fracs = sorted({r["frac"] for r in diff_rows})

fig, ax = plt.subplots(figsize=(10, 5))
for fi, feat_id in enumerate(focus_feats):
    means_B, means_C = [], []
    for frac in all_fracs:
        XB, _ = stack_group(records, groups=("B",), scope=FOCUS_SCOPE,
                            frac=frac, layer=FOCUS_LAYER, space="sae")
        XC, _ = stack_group(records, groups=("C",), scope=FOCUS_SCOPE,
                            frac=frac, layer=FOCUS_LAYER, space="sae")
        means_B.append(float(XB[:, feat_id].mean()) if XB is not None else float('nan'))
        means_C.append(float(XC[:, feat_id].mean()) if XC is not None else float('nan'))
    ax.plot(all_fracs, means_B, f"C{fi}-", label=f"feat {feat_id} B", linewidth=2)
    ax.plot(all_fracs, means_C, f"C{fi}--", label=f"feat {feat_id} C", alpha=0.6)

ax.axvline(FOCUS_FRAC, color="gray", linestyle=":", label=f"focus frac={FOCUS_FRAC}")
ax.set_xlabel("Denoising fraction (frac)"); ax.set_ylabel("Mean SAE activation")
ax.set_title(f"Top BvsC features activation over denoising steps — Layer {FOCUS_LAYER}")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig("feature_activation_over_steps.png", dpi=150)
plt.show()
print("saved feature_activation_over_steps.png")

## Phase 5：一键全流程 mechanism summary + 保存

In [ ]:
from cdg.interpret import mechanism_summary

# W_unembed 需要在上面的 Cell 里已加载
summary = mechanism_summary(
    records, sae, tokenizer, W_unembed,
    scope=FOCUS_SCOPE,
    frac=FOCUS_FRAC,
    layer=FOCUS_LAYER,
    pairs=[("B","C"), ("A","B")],
    topk_features=TOP_K,
    topk_tokens=8,
    save_dir="./interp_output",
)
print("Saved to ./interp_output/mechanism_summary.json")
print(f"Union features: {summary['atlas']['feature_ids']}")

## 结果解读速查表

| 观察 | 含义 | 行动 |
|---|---|---|
| 某 feature 在 BvsC gap 最大 | 它是「有害注入」的主要 SAE 特征 | 放入 `feature_zero_steering` |
| TOP tokens 含 inject/execute/bypass | 该 feature 代表「执行/注入」语义 | 强力候选，值得 ablation |
| cosine_sim 低（接近 0）| 各 feature 方向正交，单义 | 可以分别 zero，精准可控 |
| co-activation 高 + cosine 低 | 攻击协同机制 | 需要同时 zero 所有相关 feature |
| BvsC gap 高但 AvsB gap 低 | 攻击信号来自模板结构，不是 harmful 内容本身 | 重点用 tpl_mask scope |
| probe AUC 在 frac=0.10 最高 | 攻击在去噪 10% 时就线性可分 | 用 frac=0.10 的 diff 做 steering |